## Tool definition

In [1]:
%run langchain_common.py

C:\Users\akhawaja\git\cs4603\wk3_langchain_agents\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [5]:
tool1.invoke({"x": 467})

21.61018278497431

In [6]:
square_root.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [7]:
agent = create_agent(llm_noreason, tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [8]:
question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

pp(response['messages'][-1].content)

'The square root of 467 is approximately 21.61018278497431.'


In [9]:
pp(response['messages'])

[
    HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='8fe40aac-ecf8-438f-af19-ed1d4e872833'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 308, 'total_tokens': 336, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_ff0d4287-36f5-4ba9-b4e2-3b24892b7b0f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eedc1-9383-7993-8a4b-3217972555d2-0', tool_calls=[{'name': 'square_root', 'args': {'x': '467'}, 'id': 'call_10f1850b-c1a3-4e36-b171-8fe20d6fc313', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 308, 'output_tokens': 28, 'total_tokens': 336, 'input_token_details': {}, 'output_token_details': {}}),
    ToolMessage(content='21.61018278497431', name='square_root', id='c2be9937-

In [10]:
#Retrieve the tool calls from the LLM response
response["messages"][1].tool_calls

[{'name': 'square_root',
  'args': {'x': '467'},
  'id': 'call_10f1850b-c1a3-4e36-b171-8fe20d6fc313',
  'type': 'tool_call'}]

## create_agent (under the hood)

In [11]:
# Equivalent to create_agent, but using the bind_tools method on the LLM

llm_with_tools = llm.bind_tools(tools=[tool1])
query = "What is the square root of 900?"
response = llm_with_tools.invoke(query)
pp(response.tool_calls)

[
    {
        'args': {'x': '900'},
        'id': 'call_3f136cf0-4a68-41f0-af46-5c4ac25b5428',
        'name': 'square_root',
        'type': 'tool_call',
    },
]


## agent.invoke (under the hood)

In [12]:
query = "What is the square root of 1000?"
ai_msg = llm_with_tools.invoke(query)

pp(ai_msg)

for tool_call in ai_msg.tool_calls:
    # Look up the actual function from our 'tools' list
    # (In a real app, you'd use a dictionary mapping names to functions)
    selected_tool = {"square_root": square_root}[tool_call["name"].lower()]
    
    # 3. Execute the function with the arguments provided by the LLM
    tool_output = selected_tool.invoke(tool_call["args"])
    
    # 4. Create a ToolMessage to send the result back to the LLM
    # This must include the tool_call_id so the LLM knows which request this satisfies
    tool_message = ToolMessage(
        content=str(tool_output), 
        tool_call_id=tool_call["id"]
    )

# 5. Feed the result back to the LLM so it can give a final natural language answer
final_response = llm_with_tools.invoke([query, ai_msg, tool_message])
pp(final_response.content)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 286, 'total_tokens': 382, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen35-122b-a10b', 'system_fingerprint': None, 'id': 'chatcmpl_e4e9a915-5c90-4f84-9cd2-af35d0955a69', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eedca-b188-7a23-bad2-780bb2771ab1-0', tool_calls=[{'name': 'square_root', 'args': {'x': '1000'}, 'id': 'call_c2bf2caf-9c80-4350-9223-30f7933a35ca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 286, 'output_tokens': 96, 'total_tokens': 382, 'input_token_details': {}, 'output_token_details': {}})
[
    {
        'summary': [
            {
                'text': 'The user asked for the square root of 1000. I called the square_root function with x=1000 and received the result 31.622776601683793. I should provide this answer t